In [5]:
#!rm -rf org && git clone --filter=blob:none --sparse https://github.com/yugoguy/org.git
#!cd org && git sparse-checkout set dev/SSLVE && git checkout Latent-Variable-Evolution

Cloning into 'org'...


branch 'Latent-Variable-Evolution' set up to track 'origin/Latent-Variable-Evolution'.


Switched to a new branch 'Latent-Variable-Evolution'


In [1]:
import sys, os, glob
sys.path.insert(0, 'org/dev/SSLVE')
for f in sorted(glob.glob('org/dev/SSLVE/*.py')):
    print(f"Running: {f}")
    %run {f}

Running: org/dev/SSLVE\AgentModules.py
Running: org/dev/SSLVE\AuxLosses.py
Running: org/dev/SSLVE\BehaviorDescriptors.py
Running: org/dev/SSLVE\BehaviorMatchings.py
Running: org/dev/SSLVE\CMAME.py
Running: org/dev/SSLVE\Collectors.py
Running: org/dev/SSLVE\ExperimentUtils.py
Running: org/dev/SSLVE\LatentModules.py
Running: org/dev/SSLVE\Main.py
Running: org/dev/SSLVE\SearchPhases.py
Running: org/dev/SSLVE\VariationOperators.py


In [ ]:
#@title PlanarArmCVT LVE-ANY Multi-Seed Runner
import numpy as np
import random
import torch
import itertools

# =============================================================================
# Experiment Name (same for all settings)
# =============================================================================
EXP_NAME = 'BinPred-MixBinPred'  #@param {type:"string"}

# =============================================================================
# Sweep Settings
# =============================================================================
SEEDS = [42]
NOISE_SIGMAS = [0.0, 0.05]  # deterministic, uncertain
FITNESSES = ['angle_variance', 'sine_dependency']  # linear, non-linear

# =============================================================================
# Fixed Hyperparameters
# =============================================================================
N_JOINTS = 1000
END_EFFECTOR_DIM = 2
N_NOISE_EPISODES = 3

N_BINS = 1950
CENTERS = "Precomputed_CVT_1950"
TOP_K = 3

USE_PSE_MUT = True
USE_PSE_LINE = True
USE_LVE_MUT = True
USE_LVE_CROSS = True
USE_STD_SUPPORT_LVE = True
GREEDY_MEM = True

PSE_MUT_SIGMA = 0.05
LVE_MUT_SIGMA = 0.05
STD_SUPPORT_LO = -2.0
STD_SUPPORT_HI = 2.0

WARMUP_PSE_MUT = True
WARMUP_PSE_LINE = True

N_TOTAL = 500
WARMUP_THRESHOLD = 500
EMA_ALPHA = 0.1
TEMPERATURE = 10
MIN_PROPORTION = 0.05

USE_FLOW_PRIOR = False
LATENT_DIM = 32
HIDDEN_DIMS = [128]
BETA = 1e-2
NUM_FLOWS = 3
FLOW_HIDDEN = 128
FLOW_HIDDEN_LAYERS = 2
EPOCHS = 50
BATCH_SIZE = 512
LR = 1e-3

USE_BIN_PRED = True
GAMMA_BIN_PRED = 1e-3
USE_MIX_BIN_PRED = True
GAMMA_MIX_BIN_PRED = 1e-3
MIX_ALPHA_LO = -0.1
MIX_ALPHA_HI = 1.1

N_STEPS = 1000

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# =============================================================================
# Helpers
# =============================================================================
ANGLES_PER_JOINT = END_EFFECTOR_DIM - 1
GENE_DIM = N_JOINTS * ANGLES_PER_JOINT

def _get_metric(info, key):
    v = info[key]
    return np.mean(v) if isinstance(v, list) else v

FITNESS_FNS = {
    'angle_variance': lambda info: _get_metric(info, 'angle_variance'),
    'sine_dependency': lambda info: _get_metric(info, 'sine_dependency'),
}

FITNESS_LABELS = {
    'angle_variance': 'linear',
    'sine_dependency': 'nonlinear',
}

NOISE_LABELS = {
    0.0: 'deterministic',
    0.05: 'uncertain',
}

init_fn = lambda: np.random.uniform(-np.pi, np.pi, GENE_DIM)

# =============================================================================
# Run all settings
# =============================================================================
for noise_sigma, fitness_key, seed in itertools.product(NOISE_SIGMAS, FITNESSES, SEEDS):
    tag = f"{EXP_NAME}_{NOISE_LABELS[noise_sigma]}_{FITNESS_LABELS[fitness_key]}_seed{seed}"
    print(f"\n{'='*60}")
    print(f"  {tag}")
    print(f"{'='*60}\n")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    fitness_fn = FITNESS_FNS[fitness_key]

    collector = PlanarArmCollector(
        n_joints=N_JOINTS,
        end_effector_dim=END_EFFECTOR_DIM,
        noise_sigma=noise_sigma,
        n_episodes=N_NOISE_EPISODES,
    )
    bd = PlanarArmBD_CVT(n_bins=N_BINS, centers=CENTERS, bd_dim=END_EFFECTOR_DIM)
    bm = MAPElitesBM(behavior_descriptor=bd, fitness_fn=fitness_fn, top_k=TOP_K, max_fitness=100)

    operators = []
    translate_fn = None
    if USE_PSE_MUT:
        operators.append(PSEMut(sigma=PSE_MUT_SIGMA, greedy_mem=GREEDY_MEM))
    if USE_PSE_LINE:
        operators.append(PSELine(greedy_mem=GREEDY_MEM))
    if USE_LVE_MUT:
        operators.append(LVEMut(sigma=LVE_MUT_SIGMA, greedy_mem=GREEDY_MEM))
    if USE_LVE_CROSS:
        operators.append(LVECross(greedy_mem=GREEDY_MEM))

    warmup_operators = []
    if WARMUP_PSE_MUT:
        warmup_operators.append(PSEMut(sigma=PSE_MUT_SIGMA, greedy_mem=GREEDY_MEM))
    if WARMUP_PSE_LINE:
        warmup_operators.append(PSELine(greedy_mem=GREEDY_MEM))

    aux_losses = []
    if USE_BIN_PRED:
        aux = BinPred(behavior_descriptor=bd, latent_dim=LATENT_DIM, output_dim=END_EFFECTOR_DIM)
        aux_losses.append((GAMMA_BIN_PRED, aux))
    if USE_MIX_BIN_PRED:
        mix_aux = MixBinPred(
            behavior_descriptor=bd, latent_dim=LATENT_DIM, output_dim=END_EFFECTOR_DIM,
            alpha_lo=MIX_ALPHA_LO, alpha_hi=MIX_ALPHA_HI,
        )
        aux_losses.append((GAMMA_MIX_BIN_PRED, mix_aux))

    if USE_FLOW_PRIOR:
        lm = BaseFlowVAE(
            input_dim=GENE_DIM, latent_dim=LATENT_DIM, hidden_dims=HIDDEN_DIMS,
            beta=BETA, num_flows=NUM_FLOWS, flow_hidden=FLOW_HIDDEN,
            flow_hidden_layers=FLOW_HIDDEN_LAYERS, aux_losses=aux_losses,
        )
        translate_fn = lm.translate
        if USE_MIX_BIN_PRED:
            mix_aux.to_base_fn = lambda z: lm.flow.f(z)[0]
            mix_aux.from_base_fn = lm.flow.f_inv
    else:
        lm = BaseBetaVAE(
            input_dim=GENE_DIM, latent_dim=LATENT_DIM, hidden_dims=HIDDEN_DIMS,
            beta=BETA, aux_losses=aux_losses,
        )

    if USE_STD_SUPPORT_LVE:
        operators.append(StandardNormalSupportLVE(lo=STD_SUPPORT_LO, hi=STD_SUPPORT_HI, translate_fn=translate_fn))

    sp = BoltzmannMix(
        agent_class=PlanarArmAgent, architecture=GENE_DIM,
        operators=operators, warmup_operators=warmup_operators,
        n_total=N_TOTAL, warmup_threshold=WARMUP_THRESHOLD,
        ema_alpha=EMA_ALPHA, temperature=TEMPERATURE,
        min_proportion=MIN_PROPORTION, init_fn=init_fn,
    )

    orchestrator = SSLVE(
        search_phase=sp, collector=collector,
        behavior_matching=bm, latent_module=lm, device=DEVICE,
    )

    print(f"Noise sigma: {noise_sigma}, Fitness: {fitness_key}, Seed: {seed}")
    print(f"Latent dim: {LATENT_DIM}, Hidden: {HIDDEN_DIMS}, Beta: {BETA}")
    print(f"Flow: {USE_FLOW_PRIOR}, BinPred: {USE_BIN_PRED}, MixBinPred: {USE_MIX_BIN_PRED}")
    print(f"Operators: {[op.name for op in operators]}")
    print()

    train_kwargs = {
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'lr': LR, 'verbose': True,
    }
    histories = orchestrator.run(n_steps=N_STEPS, train_kwargs=train_kwargs)

    f_min, f_mean, f_max = bm.fitness_stats()
    print(f"\nFinal archive size: {bm.archive_size()}")
    print(f"Final coverage: {bm.coverage():.4f}")
    print(f"Best fitness ({fitness_key}): {f_min:.4f}")
    print(f"QD-score: {bm.qd_score():.4f}")

    save_path = f"./archive/{tag}/"
    save_checkpoint(save_path, bm, orchestrator.history, sp=sp, lm=lm)
    print(f"Saved to {save_path}")


  BinPred-MixBinPred_deterministic_linear_seed42

Noise sigma: 0.0, Fitness: angle_variance, Seed: 42
Latent dim: 32, Hidden: [128], Beta: 0.01
Flow: False, BinPred: True, MixBinPred: True
Operators: ['pse_mut', 'pse_line', 'lve_mut', 'lve_cross', 'std_support_lve']


--- SSLVE Step 1/1000 ---
  Operator ratio - pse_mut: 500/500 (100%), pse_line: 0/500 (0%)
Collecting: 500/500 [4s elapsed, 0s remaining]
Archive: 36, Bins: 14, Coverage: 0.0072, Fitness min/mean/max: 2.98/3.18/3.45
QD-score: 1355.7336

--- SSLVE Step 2/1000 ---
  Operator ratio - pse_mut: 250/500 (50%), pse_line: 250/500 (50%)
Collecting: 500/500 [4s elapsed, 0s remaining]
Archive: 55, Bins: 20, Coverage: 0.0103, Fitness min/mean/max: 1.42/1.80/3.37
QD-score: 1968.1782

--- SSLVE Step 3/1000 ---
  Operator ratio - pse_mut: 250/500 (50%), pse_line: 250/500 (50%)
Collecting: 500/500 [4s elapsed, 0s remaining]
Archive: 94, Bins: 40, Coverage: 0.0205, Fitness min/mean/max: 0.74/1.08/1.84
QD-score: 3959.9190

--- SSLVE Step 